## SETUP

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

In [ ]:
print(f'PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.11.0+cpu | CUDA available: False


: 

In [3]:
# Configuração centralizada
# Todos os paths, hiperparâmetros e flags ficam aqui, para facilitar manutenção e evitar hardcoding espalhado pelo código

DATA_DIR            = Path(r"saida")
DATA_DIR_FEATURE    = DATA_DIR / "feature_extraction" # tem os _reps
OUT_REPS_LABELED    = DATA_DIR / "feature_extraction_labeled/reps_labeled_all.csv"
DATA_DIR_LANDMARKS  = DATA_DIR / "landmarks" # tem os _landmarks
DATA_DIR_RESULTADOS = DATA_DIR / "resultados"
LABELS_PATH         = Path(r"rotulo") / "labels_falha.csv"     

# Identificadores de data augmentation
AUG_PREFIXES = ("aug_brightness_", "aug_combined_", "aug_flip_", "aug_time_warp_")

# Features de contexto por repetição (biomecânicas e temporais)
REP_META_COLS = ["rep_duration", "rom_deg", "elbow_start_deg", "elbow_peak_deg", "mean_vis_arm"]

# Reprodutibilidade
SEED = 2609

# Dispositivo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hiperparâmetros do modelo
MODEL_CFG = dict(
    hidden_dim  = 128,      # Dimensão do LSTM
    num_layers  = 2,        # Camadas LSTM
    dropout     = 0.4,      # Dropout geral
    ctx_hidden  = 64,       # MLP de contexto
)

# Hiperparâmetros de treinamento
TRAIN_CFG = dict(
    batch_size   = 64,
    epochs       = 50,
    lr           = 1e-3,
    lr_decay_min = 0.01,   
    patience     = 10,
    grad_clip    = 1.0,
    weight_decay = 1e-4,
    
    # Focal Loss
    focal_gamma  = 2.0,     
    focal_alpha_min = 0.25, # alpha = max(alpha_min, 1 - pos_rate) para lidar com desbalanceamento


    # Mixup augmentation (no espaço de features)
    mixup_alpha = 0.2,      # 0 = desligamento, >0 = beta(alpha, alpha) para amostrar o mixup

)

# Cross validation
CV_CFG = dict(
    n_splits = 5,
    shuffle = True,
)


np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | Seed: {SEED}")
print(f"Model: hidden_dim={MODEL_CFG['hidden_dim']}, num_layers={MODEL_CFG['num_layers']}, dropout={MODEL_CFG['dropout']}")
print(f"Training: batch_size={TRAIN_CFG['batch_size']}, epochs={TRAIN_CFG['epochs']}, lr={TRAIN_CFG['lr']}, patience={TRAIN_CFG['patience']}")
print(f"Cross validation: n_splits={CV_CFG['n_splits']}, shuffle={CV_CFG['shuffle']}")

Device: cuda | Seed: 2609
Model: hidden_dim=128, num_layers=2, dropout=0.4
Training: batch_size=64, epochs=50, lr=0.001, patience=10
Cross validation: n_splits=5, shuffle=True


## Geração de Repetições com Label

In [4]:
def get_parent_video_id(video_id: str) -> str:
    for p in AUG_PREFIXES:
        if video_id.startswith(p):
            return video_id[len(p):]
    return video_id

labels = pd.read_csv(LABELS_PATH)  # video_id, fail_rep_id
labels["fail_rep_id"] = pd.to_numeric(labels["fail_rep_id"], errors="coerce")

all_reps = []
for f in DATA_DIR_FEATURE.glob("*_reps.csv"):
    video_id = f.name.replace("_reps.csv", "")
    r = pd.read_csv(f)
    r["video_id"] = video_id
    r["parent_video_id"] = get_parent_video_id(video_id)

    # tipos numéricos
    for c in ["rep_id","start_frame","end_frame","start_time","end_time",
              "rep_duration","rom_deg","elbow_start_deg","elbow_peak_deg","mean_vis_arm"]:
        if c in r.columns:
            r[c] = pd.to_numeric(r[c], errors="coerce")

    all_reps.append(r)

reps = pd.concat(all_reps, ignore_index=True)


# merge labels
df = reps.merge(labels, on="video_id", how="left")
labels_parent = labels.rename(columns={"video_id":"parent_video_id", "fail_rep_id":"fail_rep_id_parent"})
df = df.merge(labels_parent, on="parent_video_id", how="left")
df["fail_rep_id_final"] = df["fail_rep_id"].combine_first(df["fail_rep_id_parent"])
df["fail_rep_id_final"] = pd.to_numeric(df["fail_rep_id_final"], errors="coerce") 

df["fatigue_label"] = np.where(df["fail_rep_id_final"].notna() & (df["rep_id"] >= df["fail_rep_id_final"]), 1, 0).astype(int)

# ============ FEATURES TEMPORAIS / CONTEXTUAIS POR VÍDEO ============

# total de reps no vídeo e progresso relativo (0..1) 
df["total_reps"] = df.groupby("video_id")["rep_id"].transform("max") + 1
df["rep_progress"] = df["rep_id"] / df["total_reps"].clip(lower=1)  # evita divisão por zero

# delta features: variação em relação à rep anterior (sinais de degregação)
for col in REP_META_COLS:
    if col in df.columns:
        df[f"delta_{col}"] = df.groupby("video_id")[col].diff().fillna(0)

# features relativas à primeira rep (baseline): quanto mudou desde o início
for col in REP_META_COLS:
    if col in df.columns:
        first_val = df.groupby("video_id")[col].transform("first")
        df[f"rel_{col}"] = (df[col] - first_val) / first_val.clip(lower=1e-6)

# média mível das últimas 3 reps (suaviza ruido)
for col in REP_META_COLS:
    if col in df.columns:
        df[f"rolling3_{col}"] = df.groupby("video_id")[col].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

df.to_csv(OUT_REPS_LABELED, index=False)
print("OK ->", OUT_REPS_LABELED)
print(f"Shape: {df.shape}, Positive rate {df['fatigue_label'].mean():.3%}")
df[["video_id","rep_id","fatigue_label","rep_progress","rom_deg","delta_rom_deg","rel_rom_deg"]].head(10)

OK -> saida\feature_extraction_labeled\reps_labeled_all.csv
Shape: (2877, 39), Positive rate 26.347%


,video_id,rep_id,fatigue_label,rep_progress,rom_deg,delta_rom_deg,rel_rom_deg
0,aug_brightness_friends_record_01,0,0,0.000000,54.370219,0.000000,0.000000
1,aug_brightness_friends_record_01,1,0,0.027778,155.541512,101.171292,1.860785
2,aug_brightness_friends_record_01,2,0,0.055556,35.499726,-120.041785,-0.347074
3,aug_brightness_friends_record_01,3,0,0.083333,64.519755,29.020029,0.186675
4,aug_brightness_friends_record_01,4,0,0.111111,112.351931,47.832176,1.066424
5,aug_brightness_friends_record_01,5,0,0.138889,71.678770,-40.673161,0.318346
6,aug_brightness_friends_record_01,6,0,0.166667,155.687204,84.008434,1.863465
7,aug_brightness_friends_record_01,7,0,0.194444,28.761796,-126.925408,-0.471001
8,aug_brightness_friends_record_01,8,0,0.222222,99.871818,71.110022,0.836885
9,aug_brightness_friends_record_01,9,0,0.250000,163.807423,63.935605,2.012815


## Escolha T

In [5]:
df = pd.read_csv(OUT_REPS_LABELED)

rep_len = (df["end_frame"] - df["start_frame"] + 1).clip(lower=1)
p50, p75, p90 = np.percentile(rep_len, [50, 75, 90]).astype(int)

T = int(p75)                 
T = int(np.clip(T, 30, 120))

print("frames/rep: median=", p50, "p75=", p75, "p90=", p90)
print("T escolhido:", T, "=>", round(T/30, 2), "s")

# ====== Define as colunas de feature de contexto (rep_level) ======
CONTEXT_FEATURES = ["rep_progress","total_reps"]

for col in REP_META_COLS:
    for prefix in ["","delta_","rel_","rolling3_"]:
        fname = f"{prefix}{col}"
        if fname in df.columns:
            CONTEXT_FEATURES.append(fname)

CONTEXT_FEATURES = [c for c in dict.fromkeys(CONTEXT_FEATURES) if c in df.columns]
print(f"Context features ({len(CONTEXT_FEATURES)}):", CONTEXT_FEATURES)

frames/rep: median= 60 p75= 70 p90= 88
T escolhido: 70 => 2.33 s
Context features (22): ['rep_progress', 'total_reps', 'rep_duration', 'delta_rep_duration', 'rel_rep_duration', 'rolling3_rep_duration', 'rom_deg', 'delta_rom_deg', 'rel_rom_deg', 'rolling3_rom_deg', 'elbow_start_deg', 'delta_elbow_start_deg', 'rel_elbow_start_deg', 'rolling3_elbow_start_deg', 'elbow_peak_deg', 'delta_elbow_peak_deg', 'rel_elbow_peak_deg', 'rolling3_elbow_peak_deg', 'mean_vis_arm', 'delta_mean_vis_arm', 'rel_mean_vis_arm', 'rolling3_mean_vis_arm']


## Loader de Landmarks + Colunas de Feature

In [6]:
def load_landmarks(video_id: str, cache: dict):
    if video_id in cache:
        return cache[video_id]
    f = DATA_DIR_LANDMARKS / f"{video_id}_landmarks.csv"
    lm = pd.read_csv(f)
    lm["frame"] = pd.to_numeric(lm["frame"], errors="coerce").fillna(0).astype(int)
    cache[video_id] = lm
    return lm

def feature_cols_from_landmarks(lm_df: pd.DataFrame):
    return [c for c in lm_df.columns if c not in ("frame", "time_sec")]

# pega um exemplo pra definir feature_cols/col_index
sample_video = df["video_id"].iloc[0]
lm_sample = pd.read_csv(DATA_DIR_LANDMARKS / f"{sample_video}_landmarks.csv")
feature_cols = feature_cols_from_landmarks(lm_sample)
col_index = {c:i for i,c in enumerate(feature_cols)}

print(f"Landmark feature columns ({len(feature_cols)}):", feature_cols[:6])
print(f"Context features: {len(CONTEXT_FEATURES)}", CONTEXT_FEATURES[:6])

Landmark feature columns (32): ['lm23_x', 'lm23_y', 'lm23_z', 'lm23_vis', 'lm24_x', 'lm24_y']
Context features: 22 ['rep_progress', 'total_reps', 'rep_duration', 'delta_rep_duration', 'rel_rep_duration', 'rolling3_rep_duration']


## Pré-processamento

In [7]:
def resample_to_T(X, T):
    # X: [N,F] -> [T,F] via interpolação
    N, F = X.shape
    if N == T:
        return X.astype(np.float32)
    if N < 2:
        return np.repeat(X, T, axis=0).astype(np.float32)

    old = np.linspace(0, 1, N)
    new = np.linspace(0, 1, T)

    Xr = np.zeros((T, F), dtype=np.float32)
    for j in range(F):
        Xr[:, j] = np.interp(new, old, X[:, j])
    return Xr

def swap_lr_landmarks(X, col_index, lmA, lmB):
    # troca todos os eixos x,y,z,vis entre lmA e lmB (ex: 11 <-> 12)
    for suffix in ["x","y","z","vis"]:
        a = f"lm{lmA}_{suffix}"
        b = f"lm{lmB}_{suffix}"
        if a in col_index and b in col_index:
            ia, ib = col_index[a], col_index[b]
            X[:, [ia, ib]] = X[:, [ib, ia]]

def canonicalize_side(X, col_index, side):
    """
    Canoniza para reduzir variância:
    - Se RIGHT: espelha x (x -> 1-x) + troca pares L/R (11<->12, 13<->14, 15<->16, 23<->24)
      Assim, o movimento do braço ativo tende a ficar sempre no mesmo "lado canônico".
    """
    if side is None or str(side).upper() == "NAN":
        return X
    side = str(side).upper()
    if side != "RIGHT":
        return X

    # espelha x
    for lm in [11,12,13,14,15,16,23,24]:
        k = f"lm{lm}_x"
        if k in col_index:
            X[:, col_index[k]] = 1.0 - X[:, col_index[k]]

    # troca pares L/R
    swap_lr_landmarks(X, col_index, 11, 12)
    swap_lr_landmarks(X, col_index, 13, 14)
    swap_lr_landmarks(X, col_index, 15, 16)
    swap_lr_landmarks(X, col_index, 23, 24)

    return X

def normalize_body_framewise_xyz(X, col_index):
    """
    Centraliza por mid-hip (23/24) e escala por shoulder width (11/12).
    Aplica apenas em x,y,z (não mexe em vis).
    """
    # origem: mid-hip
    hx = (X[:, col_index["lm23_x"]] + X[:, col_index["lm24_x"]]) / 2
    hy = (X[:, col_index["lm23_y"]] + X[:, col_index["lm24_y"]]) / 2
    hz = (X[:, col_index["lm23_z"]] + X[:, col_index["lm24_z"]]) / 2

    dx = X[:, col_index["lm11_x"]] - X[:, col_index["lm12_x"]]
    dy = X[:, col_index["lm11_y"]] - X[:, col_index["lm12_y"]]
    dz = X[:, col_index["lm11_z"]] - X[:, col_index["lm12_z"]]
    scale = np.sqrt(dx*dx + dy*dy + dz*dz)
    scale = np.clip(scale, 1e-3, None)

    for lm in [11,12,13,14,15,16,23,24]:
        for ax, h in [("x", hx), ("y", hy), ("z", hz)]:
            k = f"lm{lm}_{ax}"
            if k in col_index:
                X[:, col_index[k]] = (X[:, col_index[k]] - h) / scale

    return X

def add_velocity_acceleration(X):
    """ Adiciona velocidade (1ª derivada) e aceleração (2ª derivada) como canais extras.
    X: [T, F] -> Retorna [T, F*3] (posição + velocidade + aceleração)"""
    vel = np.zeros_like(X)
    acc = np.zeros_like(X)
    vel[1:] = X[1:] - X[:-1]
    acc[2:] = vel[2:] - vel[1:-1]
    return np.concatenate([X, vel, acc], axis=1)

def build_rep_sequence(lm_df, feature_cols, start_frame, end_frame, T, col_index, side=None):
    seg = lm_df[(lm_df["frame"] >= start_frame) & (lm_df["frame"] <= end_frame)].sort_values("frame")

    if len(seg) == 0:
        return np.zeros((T, len(feature_cols)), dtype=np.float32)

    # garante numérico + interpola buracos (frames sem pose) e preenche bordas
    feats = seg[feature_cols].apply(pd.to_numeric, errors="coerce")
    feats = feats.interpolate(limit_direction="both")   # linear dentro da rep
    feats = feats.fillna(0.0)                           # se ainda sobrar NaN, zera

    X = feats.to_numpy(dtype=np.float32)

    # segurança extra
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X = resample_to_T(X, T)
    X = canonicalize_side(X, col_index, side)
    X = normalize_body_framewise_xyz(X, col_index)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X = add_velocity_acceleration(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    return X

## Normalização por FOLD

In [8]:
def streaming_mean_std(sequences_iter):
    """
    sequences_iter: iterável de arrays [T,F]
    Retorna mean[F], std[F] usando Welford (sem guardar tudo em memória).
    """
    n = 0
    mean = None
    M2 = None

    for X in sequences_iter:
        # X: [T,F] -> soma em T
        # processa linha a linha (timesteps) para exatidão
        for row in X:
            n += 1
            if mean is None:
                mean = row.astype(np.float64).copy()
                M2 = np.zeros_like(mean)
            else:
                delta = row - mean
                mean += delta / n
                delta2 = row - mean
                M2 += delta * delta2

    if n < 2:
        std = np.ones_like(mean)
    else:
        var = M2 / (n - 1)
        std = np.sqrt(np.maximum(var, 1e-12))

    return mean.astype(np.float32), std.astype(np.float32)

def compute_context_stats(train_df, context_features):
    """Calcula mean/std das context features para normalização."""
    vals = train_df[context_features].apply(pd.to_numeric, errors="coerce").fillna(0).values
    mean = vals.mean(axis=0).astype(np.float32)
    std = vals.std(axis=0).astype(np.float32)
    std = np.clip(std, 1e-6, None)  # evita divisão por zero
    return mean, std

## Dataset + DataLoader

In [9]:
class RepDataset(Dataset):
    def __init__(self, reps_df, feature_cols, T, col_index, context_features, ctx_mean=None, ctx_std=None, seq_mean=None, seq_std=None):
        self.df = reps_df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.T = T
        self.col_index = col_index
        self.context_features = context_features
        self.ctx_mean = ctx_mean
        self.ctx_std = ctx_std
        self.seq_mean = seq_mean
        self.seq_std = seq_std
        self.cache = {}  # cache de landmarks por dataset (por fold)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        video_id = r["video_id"]
        side = r.get("side", None)
        start_f = int(r["start_frame"])
        end_f = int(r["end_frame"])
        y = int(r["fatigue_label"])

        # Landmark sequence [T, F*3]
        lm = load_landmarks(video_id, self.cache)
        X = build_rep_sequence(lm, self.feature_cols, start_f, end_f, self.T, self.col_index, side=side)

        # z-score (fold)
        if self.seq_mean is not None and self.seq_std is not None:
            X = (X - self.seq_mean) / (self.seq_std + 1e-8)

        # Context features (rep-level)
        ctx_vals = []
        for c in self.context_features:
            v = r.get(c, 0)
            ctx_vals.append(float(v) if pd.notna(v) else 0.0)
        ctx = np.array(ctx_vals, dtype=np.float32)

        if self.ctx_mean is not None and self.ctx_std is not None:
            ctx = (ctx - self.ctx_mean) / (self.ctx_std + 1e-8)

        return (torch.from_numpy(X), 
                torch.from_numpy(ctx),
                torch.tensor(y, dtype=torch.float32))

## Modelo: BiLSTM com Atenção + Features de Contexto

In [10]:
class TemporalAttention(nn.Module):
    """ Attention over LSTM timesteps - aprende quais frames são mais informativos."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1, bias=False)
        )

    def forward(self, lstm_out):
        # lstm_out: [B, T, H]
        scores = self.attn(lstm_out).squeeze(-1) # [B, T]
        weights = torch.softmax(scores, dim=1)   # [B, T]
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1) # [B, H]
        return context, weights

In [11]:
class FatigueNet(nn.Module):
    """
    Multi-input BiLSTM + Attention + Context Features.
    - Ramo 1: BiLSTM com atenção sobre a sequência de landmarks (pos+vel+acc)
    - Ramo 2: MLP sobre features de contexto da rep (biomecânicas + temporais)
    - Fusão: concatenção + classificador final
    """

    def __init__(self, seq_input_dim, ctx_input_dim, hidden_dim=128, num_layers=2, dropout=0.4, ctx_hidden=64):
        super().__init__()

        # Ramo sequencial
        self.lstm = nn.LSTM(
            input_size=seq_input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        self.attention = TemporalAttention(hidden_dim * 2)
        self.seq_norm = nn.LayerNorm(hidden_dim * 2)

        # Ramo de contexto
        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_input_dim, ctx_hidden),
            nn.ReLU(),
            nn.Dropout(dropout*0.5),
            nn.Linear(ctx_hidden, ctx_hidden//2),
            nn.ReLU(),
        )

        # Fusão + Classificador
        fusion_dim = hidden_dim * 2 + ctx_hidden//2
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_ctx):
        # x_seq: [B, T, F*3], x_ctx: [B, C]
        lstm_out, _ = self.lstm(x_seq)  # [B, T, H*2]
        attn_out, _ = self.attention(lstm_out)  # [B, H*2]
        attn_out = self.seq_norm(attn_out)

        ctx_out = self.ctx_mlp(x_ctx)  # [B, 32]

        fused = torch.cat([attn_out, ctx_out], dim=1)  # [B, H*2 + 32]
        logits = self.classifier(fused).squeeze(1)  # [B]
        return logits

## Treino, Métricas e Treshold por fold

In [12]:
class FocalLoss(nn.Module):
    """ Focal Loss para lidar com desbalanceamento. """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()

def metrics_from_probs(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    out = {
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
    }
    return out

def best_threshold_for_f1(y_true, y_prob, n_steps=50):
    best_thr, best_f1 = 0.5, -1
    for thr in np.linspace(0.1, 0.9, n_steps):
        f1 = metrics_from_probs(y_true, y_prob, thr)["f1"]
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr, best_f1

def eval_loader(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for Xb, ctx_b, yb in loader:
            Xb = Xb.to(device)
            ctx_b = ctx_b.to(device)
            logits = model(Xb, ctx_b)
            prob = torch.sigmoid(logits).cpu().numpy()
            ps.append(prob)
            ys.append(yb.numpy())
    y_true = np.concatenate(ys).astype(int)
    y_prob = np.concatenate(ps)
    return y_true, y_prob


def train_one_fold(train_df, val_df, feature_cols, T, col_index, context_features):
    
    cfg = TRAIN_CFG
    mcfg = MODEL_CFG

    # ==== calcula mean/std do treino (sem vazamento) ====
    tmp_cache = {}
    def seq_iter():
        for _, r in train_df.iterrows():
            lm = load_landmarks(r["video_id"], tmp_cache)
            yield build_rep_sequence(
                lm, feature_cols,
                int(r["start_frame"]), int(r["end_frame"]),
                T, col_index, side=r.get("side", None)
            )

    seq_mean, seq_std = streaming_mean_std(seq_iter())

    # Normalização das context features
    ctx_mean, ctx_std = compute_context_stats(train_df, context_features)

    # datasets
    train_ds = RepDataset(train_df, feature_cols, T, col_index, context_features, ctx_mean, ctx_std, seq_mean, seq_std)
    val_ds   = RepDataset(val_df, feature_cols, T, col_index, context_features, ctx_mean, ctx_std, seq_mean, seq_std)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

    # Modelo
    seq_input_dim = len(feature_cols) * 3  # pos + vel + acc
    ctx_input_dim = len(context_features)

    model = FatigueNet(
        seq_input_dim=seq_input_dim,
        ctx_input_dim=ctx_input_dim,
        hidden_dim=mcfg["hidden_dim"],
        num_layers=mcfg["num_layers"],
        dropout=mcfg["dropout"]
    ).to(DEVICE)
    
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["lr"]*cfg["lr_decay_min"])

    # Focal Loss (melhor para desbalanceamento)
    ytr = train_df["fatigue_label"].values
    pos_rate = ytr.mean()
    crit = FocalLoss(alpha=max(cfg["focal_alpha_min"], 1-pos_rate), gamma=cfg["focal_gamma"])

    best_state = None
    best_val_f1 = -1
    best_thr = 0.5
    bad = 0

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        losses = []

        for Xb, ctx_b, yb in train_loader:
            Xb = Xb.to(DEVICE)
            ctx_b = ctx_b.to(DEVICE)
            yb = yb.to(DEVICE)

            opt.zero_grad()
            logits = model(Xb, ctx_b)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["grad_clip"])
            opt.step()
            losses.append(loss.item())

        scheduler.step()

        # Validação
        y_true, y_prob = eval_loader(model, val_loader, DEVICE)
        thr, f1_val = best_threshold_for_f1(y_true, y_prob)
        m = metrics_from_probs(y_true, y_prob, thr)

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"epoch {epoch:02d} | loss={np.mean(losses):.4f} | lr={scheduler.get_last_lr()[0]:.6f} | "
                f"val_f1={m['f1']:.3f} (thr={thr:.2f}) acc={m['acc']:.3f} "
                f"prec={m['precision']:.3f} rec={m['recall']:.3f} auc={m['roc_auc']:.3f}"
            )

        # early stopping por F1
        if m["f1"] > best_val_f1:
            best_val_f1 = m["f1"]
            best_thr = thr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch} with best val F1={best_val_f1:.3f} at thr={best_thr:.2f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, seq_mean, seq_std, ctx_mean, ctx_std, best_thr, best_val_f1

## Cross-Validation

In [13]:
df = pd.read_csv(OUT_REPS_LABELED)

# sanitiza
df["fatigue_label"] = pd.to_numeric(df["fatigue_label"], errors="coerce").fillna(0).astype(int)
df["rep_id"] = pd.to_numeric(df["rep_id"], errors="coerce")
df["start_frame"] = pd.to_numeric(df["start_frame"], errors="coerce")
df["end_frame"] = pd.to_numeric(df["end_frame"], errors="coerce")
df["parent_video_id"] = df["parent_video_id"].astype(str)
df["side"] = df["side"].astype(str)

for c in CONTEXT_FEATURES:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        
X = df
y = df["fatigue_label"].values
groups = df["parent_video_id"].values

cv = StratifiedGroupKFold(n_splits=CV_CFG["n_splits"], shuffle=CV_CFG["shuffle"], random_state=SEED)

fold_rows = []
all_preds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups), 1):
    train_df = df.iloc[tr_idx].copy()
    val_df   = df.iloc[va_idx].copy()

    # anti-vazamento
    assert set(train_df["parent_video_id"]).isdisjoint(set(val_df["parent_video_id"]))

    print(f"\n{'='*60}")
    print(f"\n==== FOLD {fold}/5 ====")
    print("train parents:", train_df["parent_video_id"].nunique(),
          "| val parents:", val_df["parent_video_id"].nunique(),
          "| train pos_rate:", round(train_df["fatigue_label"].mean(), 3),
          "| val pos_rate:", round(val_df["fatigue_label"].mean(), 3))

    model, seq_mean, seq_std, ctx_mean, ctx_std, thr, best_f1 = train_one_fold(
        train_df, val_df,
        feature_cols=feature_cols,
        T=T,
        col_index=col_index,
        context_features=CONTEXT_FEATURES
    )

    # avaliação final do fold (no melhor estado)
    val_ds = RepDataset(val_df, feature_cols, T, col_index, CONTEXT_FEATURES, ctx_mean, ctx_std, seq_mean, seq_std)
    val_loader = DataLoader(val_ds, batch_size=TRAIN_CFG["batch_size"], shuffle=False, num_workers=0)

    y_true, y_prob = eval_loader(model, val_loader, DEVICE)
    m = metrics_from_probs(y_true, y_prob, thr)

    ckpt = {
        "state_dict": model.state_dict(),
        "seq_mean": seq_mean,
        "seq_std": seq_std,
        "ctx_mean": ctx_mean,
        "ctx_std": ctx_std,
        "threshold": thr,
        "T": T,
        "feature_cols": feature_cols,
        "context_features": CONTEXT_FEATURES,
        "fold": fold
    }
    torch.save(ckpt, DATA_DIR_RESULTADOS / f"model_fold{fold}.pth")

    fold_rows.append({"fold": fold, "thr": thr, **m})
    
    # Store predictions for the current fold
    fold_preds = pd.DataFrame({
        "fold": fold,
        "video_id": val_df["video_id"].values,
        "rep_id": val_df["rep_id"].values,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": (y_prob >= thr).astype(int)
    })
    all_preds.append(fold_preds)

results = pd.DataFrame(fold_rows)
results.to_csv(DATA_DIR_RESULTADOS / "cv_results.csv", index=False)

preds_df = pd.concat(all_preds, ignore_index=True)
preds_df.to_csv(DATA_DIR_RESULTADOS / "cv_predictions.csv", index=False)

results



==== FOLD 1/5 ====
train parents: 44 | val parents: 10 | train pos_rate: 0.264 | val pos_rate: 0.261
epoch 01 | loss=0.0687 | lr=0.000999 | val_f1=0.417 (thr=0.36) acc=0.270 prec=0.263 rec=1.000 auc=0.576
epoch 05 | loss=0.0244 | lr=0.000976 | val_f1=0.738 (thr=0.39) acc=0.841 prec=0.646 rec=0.859 auc=0.898
epoch 10 | loss=0.0138 | lr=0.000905 | val_f1=0.806 (thr=0.43) acc=0.888 prec=0.735 rec=0.893 auc=0.924
epoch 15 | loss=0.0094 | lr=0.000796 | val_f1=0.778 (thr=0.26) acc=0.870 prec=0.703 rec=0.872 auc=0.912
epoch 20 | loss=0.0073 | lr=0.000658 | val_f1=0.778 (thr=0.23) acc=0.872 prec=0.711 rec=0.859 auc=0.923
Early stopping at epoch 20 with best val F1=0.806 at thr=0.43


==== FOLD 2/5 ====
train parents: 42 | val parents: 12 | train pos_rate: 0.264 | val pos_rate: 0.261
epoch 01 | loss=0.0716 | lr=0.000999 | val_f1=0.483 (thr=0.48) acc=0.546 prec=0.344 rec=0.813 auc=0.634
epoch 05 | loss=0.0259 | lr=0.000976 | val_f1=0.848 (thr=0.46) acc=0.913 prec=0.781 rec=0.927 auc=0.968
epoc

,fold,thr,acc,f1,precision,recall,roc_auc,pr_auc
0,1,0.426531,0.887916,0.806061,0.734807,0.892617,0.923789,0.765236
1,2,0.573469,0.947826,0.901961,0.884615,0.920000,0.981663,0.954816
2,3,0.720408,0.930192,0.866667,0.884354,0.849673,0.968098,0.927894
3,4,0.622449,0.908621,0.841791,0.805714,0.881250,0.959628,0.906418
4,5,0.606122,0.897924,0.806557,0.773585,0.842466,0.952087,0.877431


In [14]:
print(f"\n{'='*60}")
print("\n==== MÉDIA ± STD (5 folds) ====")
print(f"{'='*60}")
for col in ["acc","f1","precision","recall","roc_auc","pr_auc"]:
    mean = results[col].mean()
    std  = results[col].std()
    print(f"{col:10s}: {mean:.4f} ± {std:.4f}")
print("\nArquivos salvos em:", DATA_DIR_RESULTADOS)
print("- cv_results.csv")
print("- cv_predictions.csv")
print("- model_fold1.pth ... model_fold5.pth")



==== MÉDIA ± STD (5 folds) ====
acc       : 0.9145 ± 0.0244
f1        : 0.8446 ± 0.0410
precision : 0.8166 ± 0.0668
recall    : 0.8772 ± 0.0318
roc_auc   : 0.9571 ± 0.0216
pr_auc    : 0.8864 ± 0.0734

Arquivos salvos em: saida\resultados
- cv_results.csv
- cv_predictions.csv
- model_fold1.pth ... model_fold5.pth
